# Notebook 08 — Full Map Generation

**Purpose:** Apply the best model to all SAR coverage and produce a global terrain classification map.

**Outputs:**
- `data/predictions/global_segmentation_map.tif`
- `figures/global_map.png`
- `figures/dragonfly_landing_region.png`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import rasterio
from pathlib import Path
from tqdm.notebook import tqdm

from src.utils import (
    RAW_DIR, PROCESSED_DIR, PREDICTIONS_DIR, MODELS_DIR, FIGURES_DIR,
    TERRAIN_CLASSES, CLASS_COLORS, NUM_CLASSES,
    read_geotiff, write_geotiff, get_logger,
)

log = get_logger('08_map_gen')

SAR_TILES_DIR = PROCESSED_DIR / 'sar_tiles'
LABEL_TILES_DIR = PROCESSED_DIR / 'label_tiles'

tile_df = pd.read_csv(PROCESSED_DIR / 'tile_metadata.csv')

## 1. Select Best Model

In [ ]:
# Find best model by test IoU
metric_files = sorted(MODELS_DIR.glob('*_metrics.json'))
best_model = None
best_iou = -1

for mf in metric_files:
    with open(mf) as f:
        m = json.load(f)
    
    # Handle both RF and DL metric formats
    if 'test' in m and isinstance(m['test'], dict):
        iou = m['test'].get('mean_iou', 0)
    else:
        iou = m.get('test_iou', 0)
    
    name = m.get('run_name', m.get('model', mf.stem))
    print(f"  {name}: test IoU = {iou:.4f}")
    
    if iou > best_iou:
        best_iou = iou
        best_model = name

print(f"\nBest model: {best_model} (IoU={best_iou:.4f})")

## 2. Run Inference on All Tiles

In [ ]:
# Determine if best model is RF or DL
is_rf = 'rf' in str(best_model).lower()

if is_rf:
    import joblib
    from skimage.feature import graycomatrix, graycoprops
    from scipy.stats import skew, kurtosis
    
    rf = joblib.load(MODELS_DIR / 'rf_baseline_v1.joblib')
    
    GLCM_DISTANCES = [1, 3, 5]
    GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    GLCM_PROPS = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    
    def predict_tile_rf(sar_tile):
        features = {}
        valid = sar_tile[sar_tile > 0]
        if len(valid) == 0:
            valid = sar_tile.flatten()
        features['mean'] = float(valid.mean())
        features['std'] = float(valid.std())
        features['median'] = float(np.median(valid))
        features['skewness'] = float(skew(valid))
        features['kurtosis'] = float(kurtosis(valid))
        features['p10'] = float(np.percentile(valid, 10))
        features['p90'] = float(np.percentile(valid, 90))
        
        sar_q = (np.clip(sar_tile, 0, 1) * 63).astype(np.uint8)
        glcm = graycomatrix(sar_q, distances=GLCM_DISTANCES, angles=GLCM_ANGLES,
                            levels=64, symmetric=True, normed=True)
        for prop in GLCM_PROPS:
            vals = graycoprops(glcm, prop)
            for d_idx, d in enumerate(GLCM_DISTANCES):
                features[f'{prop}_d{d}'] = float(vals[d_idx].mean())
        
        X = np.array([[features[c] for c in sorted(features.keys())]])
        return int(rf.predict(X)[0])
    
    log.info("Using RF model for map generation (tile-level predictions)")
    
else:
    # DL model
    try:
        import torch
        import segmentation_models_pytorch as smp
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = smp.Unet('resnet34', in_channels=3, classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(
            MODELS_DIR / f'{best_model}_best.pth', 
            map_location=device, weights_only=True
        ))
        model.eval()
        log.info(f"Loaded DL model: {best_model} on {device}")
    except (ImportError, FileNotFoundError) as e:
        log.warning(f"Cannot load DL model: {e}. Falling back to RF.")
        is_rf = True

In [ ]:
# Run inference on all tiles
tile_predictions = {}  # tile_id → predicted class (RF) or predicted map (DL)

all_tile_ids = sorted(tile_df['tile_id'].tolist())

for tid in tqdm(all_tile_ids, desc='Inference'):
    sar = np.load(SAR_TILES_DIR / f"{tid}.npy")
    
    if is_rf:
        pred = predict_tile_rf(sar)
        tile_predictions[tid] = pred
    else:
        with torch.no_grad():
            x = torch.from_numpy(sar).float().unsqueeze(0).unsqueeze(0).expand(1, 3, -1, -1).to(device)
            pred = model(x).argmax(dim=1).squeeze().cpu().numpy()
        tile_predictions[tid] = pred

log.info(f"Inference complete for {len(tile_predictions)} tiles")

## 3. Stitch into Global Map

In [ ]:
# Reconstruct global raster from tile predictions
TILE_SIZE = 256

# Get original raster dimensions
sar_mosaic_path = RAW_DIR / 'Titan_SAR_HiSAR_Global_Mosaic_351m.tif'
with rasterio.open(sar_mosaic_path) as src:
    full_height = src.height
    full_width = src.width
    profile = src.profile.copy()

# Create output raster
global_map = np.full((full_height, full_width), 255, dtype=np.uint8)

for _, row in tqdm(tile_df.iterrows(), total=len(tile_df), desc='Stitching'):
    tid = row['tile_id']
    if tid not in tile_predictions:
        continue
    
    r = int(row['row'])
    c = int(row['col'])
    r_start = r * TILE_SIZE
    c_start = c * TILE_SIZE
    
    pred = tile_predictions[tid]
    
    if isinstance(pred, (int, np.integer)):
        # Tile-level prediction (RF) → fill entire tile
        global_map[r_start:r_start+TILE_SIZE, c_start:c_start+TILE_SIZE] = pred
    else:
        # Pixel-level prediction (DL)
        h, w = pred.shape
        global_map[r_start:r_start+h, c_start:c_start+w] = pred.astype(np.uint8)

# Save as GeoTIFF
profile.update(dtype='uint8', count=1, nodata=255)
output_path = PREDICTIONS_DIR / 'global_segmentation_map.tif'
write_geotiff(output_path, global_map, profile, dtype='uint8')
log.info(f"Global map saved: {output_path}")

## 4. Visualise Global Map

In [ ]:
class_cmap = ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)] + ['#000000'])  # + nodata

# Downsample for display
ds_factor = max(1, global_map.shape[0] // 2000)
display_map = global_map[::ds_factor, ::ds_factor]

fig, ax = plt.subplots(figsize=(18, 9))
display_masked = np.ma.masked_where(display_map == 255, display_map)
ax.imshow(display_masked, cmap=ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)]),
          vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')

# Legend
legend_patches = [mpatches.Patch(color=CLASS_COLORS[i], label=TERRAIN_CLASSES[i]) 
                  for i in range(NUM_CLASSES)]
ax.legend(handles=legend_patches, loc='lower left', fontsize=10, framealpha=0.9)

ax.set_title(f'Titan Global Terrain Classification Map\n({best_model})', fontsize=14)
ax.set_xlabel('Pixel Column (Longitude)')
ax.set_ylabel('Pixel Row (Latitude)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'global_map.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Dragonfly Landing Region

In [ ]:
# Dragonfly landing: Shangri-La / Selk Crater
# Approximately 0-10°N, 160-200°W (or 160-200°E in 0-360 convention)
# Convert to pixel coordinates

with rasterio.open(sar_mosaic_path) as src:
    transform = src.transform
    # Convert lat/lon bounds to pixel coords
    # Note: depends on CRS — simple cylindrical typically has lon as x, lat as y
    # Approximate pixel extraction for the landing region
    total_lon_range = src.bounds.right - src.bounds.left
    total_lat_range = src.bounds.top - src.bounds.bottom
    
    # Target: lon 160-200, lat -10 to 10 (adjust based on actual CRS)
    lon_min_target, lon_max_target = 160, 200
    lat_min_target, lat_max_target = -10, 10
    
    col_start = int((lon_min_target - src.bounds.left) / total_lon_range * src.width)
    col_end = int((lon_max_target - src.bounds.left) / total_lon_range * src.width)
    row_start = int((src.bounds.top - lat_max_target) / total_lat_range * src.height)
    row_end = int((src.bounds.top - lat_min_target) / total_lat_range * src.height)
    
    # Clamp
    col_start = max(0, col_start)
    col_end = min(src.width, col_end)
    row_start = max(0, row_start)
    row_end = min(src.height, row_end)

print(f"Dragonfly region pixels: rows [{row_start}:{row_end}], cols [{col_start}:{col_end}]")

# Extract region
dragon_map = global_map[row_start:row_end, col_start:col_end]

# Also get SAR for overlay
with rasterio.open(sar_mosaic_path) as src:
    from rasterio.windows import Window
    w = Window(col_start, row_start, col_end - col_start, row_end - row_start)
    dragon_sar = src.read(1, window=w)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# SAR
valid = dragon_sar[dragon_sar > 0]
if len(valid) > 0:
    vmin, vmax = np.percentile(valid, [2, 98])
else:
    vmin, vmax = 0, 1
axes[0].imshow(dragon_sar, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('SAR Backscatter')

# Classification
dragon_masked = np.ma.masked_where(dragon_map == 255, dragon_map)
axes[1].imshow(dragon_masked, cmap=ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)]),
               vmin=0, vmax=NUM_CLASSES-1, interpolation='nearest')
axes[1].set_title('Terrain Classification')

# Overlay
axes[2].imshow(dragon_sar, cmap='gray', vmin=vmin, vmax=vmax)
axes[2].imshow(dragon_masked, cmap=ListedColormap([CLASS_COLORS[i] for i in range(NUM_CLASSES)]),
               vmin=0, vmax=NUM_CLASSES-1, alpha=0.4, interpolation='nearest')
axes[2].set_title('Overlay')

for ax in axes:
    ax.axis('off')

# Add legend
legend_patches = [mpatches.Patch(color=CLASS_COLORS[i], label=TERRAIN_CLASSES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=legend_patches, loc='lower center', ncol=NUM_CLASSES, fontsize=10)

plt.suptitle('Dragonfly Landing Region (Shangri-La / Selk Crater)', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dragonfly_landing_region.png', dpi=300, bbox_inches='tight')
plt.show()